# DAY 1: LoRA & PEFT Fundamentals

## 🎯 Learning Objectives
By the end of this day, you will be able to explain and demonstrate:
1. **Full Fine-Tuning vs. Parameter-Efficient Fine-Tuning (PEFT)**: Why training all 1.5B–70B parameters is computationally prohibitive and prone to catastrophic forgetting.
2. **The Mathematics of LoRA**: How Low-Rank Adaptation decomposes weight updates $\Delta W$ into low-rank matrices $B \cdot A$.
3. **Freezing Base Weights**: Why $W_0$ remains strictly frozen in memory while only $A$ and $B$ receive gradients.
4. **Hyperparameters**: Rank $r$, scaling factor $\alpha$, dropout, and target module selection (`q_proj`, `v_proj`, `k_proj`, `o_proj`, etc.).
5. **Parameter Efficiency**: Numerically calculating trainable parameter counts and memory savings.
6. **Adapter Saving & Loading**: Storing lightweight adapter weights (~20MB) separate from multi-gigabyte base model weights.

## 📐 Mathematical Formulation of LoRA

In standard full fine-tuning, given a pretrained weight matrix $W_0 \in \mathbb{R}^{d \times k}$, the updated weight matrix is:
$$
W' = W_0 + \Delta W
$$
where $\Delta W \in \mathbb{R}^{d \times k}$ has the exact same dimensions as $W_0$.

**LoRA Hypothesis (Hu et al., 2021)**: Weight updates during task-specific adaptation have a low **intrinsic rank** $r \ll \min(d, k)$.

LoRA parameterizes $\Delta W$ as the matrix product of two low-rank matrices:
$$
\Delta W = \left(\frac{\alpha}{r}\right) B \cdot A
$$
- $A \in \mathbb{R}^{r \times k}$ is initialized from a Gaussian distribution $\mathcal{N}(0, \sigma^2)$
- $B \in \mathbb{R}^{d \times r}$ is initialized to **zeros** (ensuring $\Delta W = 0$ at the start of training)
- $r$ is the **rank** (e.g. 8, 16, 32)
- $\alpha$ is a **constant scaling factor** (typically $2 \times r$). The scaling factor $\frac{\alpha}{r}$ stabilizes training when varying $r$.

During forward pass for input vector $x$:
$$
h = W_0 x + \Delta W x = W_0 x + \left(\frac{\alpha}{r}\right) B A x
$$

### Step 1: Environment & GPU Hardware Inspection

In [ ]:
# Install core packages if running in Google Colab
# !pip install -q transformers peft accelerate datasets bitsandbytes pyyaml

In [ ]:
import sys
import os
import random
import numpy as np
import torch
import transformers
import peft

# 1. Environment & Hardware Detection
print("=" * 60)
print("🚀 RUNTIME & HARDWARE INSPECTION")
print("=" * 60)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available       : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version         : {torch.version.cuda}")
else:
    print("GPU Device           : CPU Mode")
print(f"Transformers Version : {transformers.__version__}")
print(f"PEFT Version         : {peft.__version__}")
print("=" * 60)

# 2. Seed configuration for 100% reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Random seed locked to {SEED}.")

### Step 2: Load Model & Inspect Base Architecture

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading base model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None
)
print("Base model loaded successfully.")

### Step 3: Establish a Zero-Shot Baseline

In [ ]:
sample_prompt = "John ordered 3 laptops for $2400 and wants delivery on Friday."
messages = [
    {"role": "system", "content": 'Extract order into JSON: {"customer": str, "quantity": int, "product": str, "amount": float, "delivery_day": str}'},
    {"role": "user", "content": sample_prompt}
]

formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_input, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = base_model.generate(**inputs, max_new_tokens=100, do_sample=False)

baseline_response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("--- ZERO-SHOT BASELINE OUTPUT ---")
print(baseline_response)

### Step 4: Configure LoRA & Inspect Parameter Reduction

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,                         # LoRA Rank r
    lora_alpha=32,                # Scaling alpha
    lora_dropout=0.05,            # Dropout probability for LoRA layers
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

peft_model = get_peft_model(base_model, lora_config)

# Calculate trainable parameter stats
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in peft_model.parameters())
frozen_params = total_params - trainable_params
trainable_ratio = 100 * trainable_params / total_params

print("=" * 60)
print("📊 PARAMETER EFFICIENCY COMPARISON")
print("=" * 60)
print(f"Total Model Parameters  : {total_params:,}")
print(f"Trainable Parameters    : {trainable_params:,}")
print(f"Frozen Parameters       : {frozen_params:,}")
print(f"Trainable Percentage    : {trainable_ratio:.4f}%")
print(f"Memory Reduction Factor : {total_params / trainable_params:.1f}x fewer gradients")
print("=" * 60)

### Step 5: Save and Reload the LoRA Adapter

In [ ]:
adapter_save_path = "models/adapters/day01_lora_test"
peft_model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)

print(f"Adapter weights successfully saved to: {adapter_save_path}")

# Check disk size of adapter vs base model
adapter_files = os.listdir(adapter_save_path)
print(f"Adapter directory contents: {adapter_files}")
for f in adapter_files:
    f_size_mb = os.path.getsize(os.path.join(adapter_save_path, f)) / (1024 * 1024)
    print(f"  - {f}: {f_size_mb:.2f} MB")

---
## 🎓 Day 1 Interview Preparation & Concept Mastery

### 🧠 Concepts You Must Know
1. **Intrinsic Rank**: Over-parameterized neural networks reside on a low-dimensional manifold; hence task adaptation only needs rank $r \ll d$.
2. **Zero Initialization of Matrix B**: Matrix $B$ is initialized with 0s and $A$ with Gaussian random noise so that $\Delta W = 0$ at step 0, preserving original pretrained behavior.
3. **LoRA Scaling Factor ($\alpha / r$)**: Keeps the magnitude of weight updates stable when experimenting with different values of rank $r$.
4. **Catastrophic Forgetting Mitigation**: Because $W_0$ is frozen, the base model never loses its foundational language capabilities.
5. **Multi-Tenant Serving**: A single base model instance in GPU memory can serve dozens of task-specific LoRA adapters simultaneously by swapping or routing $\Delta W$.

### 💼 5 Interview Questions
1. **Q1 (Beginner)**: *Why does LoRA save GPU memory if the total number of forward pass parameters is identical to full fine-tuning?*
   - **Answer**: Memory savings in training do not come from the forward pass weights alone; they come from **gradients and optimizer states**. In full fine-tuning of AdamW, you store 2 optimizer states per parameter (FP32 momentum and variance = 8 bytes/param) plus gradients (2-4 bytes/param). By training only 1% of parameters, optimizer state memory drops by 99%.
2. **Q2 (Intermediate)**: *What happens if you increase rank $r$ from 16 to 128?*
   - **Answer**: Expressive capacity increases, but training memory and adapter size grow linearly. For structured tasks (like classification or JSON extraction), $r=8$ or $16$ is optimal. Higher rank risks overfitting on small datasets.
3. **Q3 (Intermediate)**: *Why do we target both attention projection modules (`q, k, v, o`) and MLP layers (`gate, up, down`)?*
   - **Answer**: Early LoRA papers only adapted $W_q$ and $W_v$. Subsequent empirical research (Dettmers et al., 2023) proved that adapting all linear layers in the transformer block achieves parity with full fine-tuning.
4. **Q4 (Advanced)**: *During inference, does LoRA introduce latency overhead? How can it be eliminated?*
   - **Answer**: If computed as $W_0 x + (\alpha/r)BAx$, there is a small matrix multiplication overhead. However, before deployment, you can permanently merge the adapter weights via $W_{\text{merged}} = W_0 + (\alpha/r)BA$ using `merge_and_unload()`, yielding **zero inference latency overhead**.
5. **Q5 (Advanced)**: *What is the difference between LoRA, Prefix Tuning, and Prompt Tuning?*
   - **Answer**: Prompt/Prefix tuning prepends learnable virtual token embeddings to keys/values, consuming context window space. LoRA directly modifies weight matrices without consuming token positions.

### 🛠️ 3 Practical Questions on Your Code
1. *In your code, what does `TaskType.CAUSAL_LM` specify to the PEFT library?*
2. *How would you calculate the theoretical optimizer memory required for full fine-tuning vs. your LoRA adapter on 1.5B parameters?*
3. *What is inside `adapter_config.json` vs `adapter_model.safetensors`?*

### 🏆 Day 1 Hands-On Challenge
> **Challenge**: Modify the `LoraConfig` to target **only** `q_proj` and `v_proj` (the original Hu et al. configuration). Run the parameter inspector and calculate the exact difference in trainable parameters and percentage compared to all-linear adaptation.